In [1]:
%%writefile sintactico_ast.py
class NodoAST:
    def traducirPy(self): 
        raise NotImplementedError('Metodo traducirPy() no implementado en este Nodo')
    def traducirPseudo_C(self): 
        raise NotImplementedError('Metodo traducirCPP() no implementado en este Nodo')
    def generarCodigo(self):
        raise NotImplementedError('Metodo generarCodigo() no implementado en este Nodo')

class NodoPrograma(NodoAST):
    def __init__(self, funcion):
        self.variables = []
        self.funcion = funcion

    def generarCodigo(self):
        codigo = ["section .text", "global _start"]
        data = ["section .bss"]
        
        main_func = next((f for f in self.funcion if f.nombre[1] == 'inicio'), None)

        # Recolectar variables
        for func in self.funcion:
            print(func.generarCodigo())

        # Punto de entrada
        codigo.append("_start:")
        if main_func:
            codigo.append("    call main")
        
        codigo.append("    mov eax, 1    ; syscall exit")
        codigo.append("    xor ebx, ebx  ; Codigo de Salida 0")
        codigo.append("    int 0x80")

        for variable in self.variables:
            if variable[0] == 'int':
                data.append(f'    {variable[1]}:    resd 1')      
        return '\n'.join(data) + '\n\n' + '\n'.join(codigo)

    def optimizar(self):
        constantes = {} 
        for f in self.funcion:
            if hasattr(f, 'optimizar'):
                f.optimizar(constantes)
        return self
            
class NodoLlamada(NodoAST):
    def __init__(self, nombre, argumentos):
        self.nombre = nombre
        self.argumentos = argumentos

    def generarCodigo(self):
        codigo = []
        for arg in reversed (self.argumentos): # Apilamos argumentos en orden inverso
            codigo.append(arg.generarCodigo())
            codigo.append("    push eax  ; Pasar argumento a la pila")

        codigo.append(f"    call {self.nombre}  : Lamar a la funcion {self.nombre}")
        codigo.append(f"    add esp, {len(self.argumentos) * 4}  ; Limpiar pila de argumentos")
        return "\n".join(codigo)

        

class NodoString(NodoAST):
    def __init__(self, valor): self.valor = valor

class NodoPrint(NodoAST):
    def __init__(self, expresiones, nueva_linea=False): 
        self.expresiones = expresiones
        self.nueva_linea = nueva_linea

    def generarCodigo(self):
        codigo = []
        for exp in self.expresiones:
            codigo.append(exp.generarCodigo())
            codigo.append("    call print_int")
        if self.nueva_linea:
            codigo.append("    call print_newline")
        return '\n'.join(codigo)

    def optimizar(self, constantes=None):
        if constantes is None: constantes = {}
        for i in range(len(self.expresiones)):
            if hasattr(self.expresiones[i], 'optimizar'):
                self.expresiones[i] = self.expresiones[i].optimizar(constantes)
        return self
        
        
class NodoFuncion(NodoAST):
    def __init__(self, tipo, nombre, parametros, cuerpo):
        self.tipo = tipo 
        self.nombre = nombre 
        self.parametros = parametros 
        self.cuerpo = cuerpo

    def generarCodigo(self):
        codigo = f'{self.nombre[1]}:\n'
        if len(self.parametros) > 0:
            for parametro in self.parametros:
                codigo += '\n    pop    eax'
                codigo += f'\n    mov [{parametro.nombre[1]}], eax'
        codigo += '\n'.join(c.generarCodigo() for c in self.cuerpo)
        codigo += '\n    ret'
        codigo += '\n'
        return codigo

    def optimizar(self, constantes=None):
        if constantes is None: constantes = {}
        nuevo_cuerpo = []
        
        for instr in self.cuerpo:
            if hasattr(instr, 'optimizar'):
                resultado = instr.optimizar(constantes)
                # Si una instrucción (como NodoSi) nos devuelve múltiples instrucciones, las integramos
                if isinstance(resultado, list):
                    nuevo_cuerpo.extend(resultado)
                else:
                    nuevo_cuerpo.append(resultado)
            else:
                nuevo_cuerpo.append(instr)
                
        self.cuerpo = nuevo_cuerpo
        return self

class NodoParametros(NodoAST):
    def __init__(self, tipo, nombre): 
        self.tipo = tipo
        self.nombre = nombre

class NodoAsignacion(NodoAST):
    def __init__(self, tipo, nombre, expresion): 
        self.tipo = tipo; self.nombre = nombre; self.expresion = expresion

    def generarCodigo(self):
        codigo = self.expresion.generarCodigo()
        codigo += f'\n    mov [{self.nombre[1]}], eax'
        return codigo

    def optimizar(self, constantes=None):
        if constantes is None: constantes = {}
        
        # Optimizamos la expresión de la derecha
        if hasattr(self.expresion, 'optimizar'):
            self.expresion = self.expresion.optimizar(constantes)
        
        # Si la expresión terminó siendo un número fijo, lo memorizamos
        if isinstance(self.expresion, NodoNumero):
            constantes[self.nombre[1]] = self.expresion.valor[1]
        elif self.nombre[1] in constantes:
            # Si se le reasigna algo no constante, lo sacamos de la memoria
            del constantes[self.nombre[1]]
            
        return self

class NodoOperacion(NodoAST):
    def __init__(self, izquierda, operador, derecha): 
        self.izquierda = izquierda; self.operador = operador; self.derecha = derecha

    def generarCodigo(self):
        codigo = []
        codigo.append(self.izquierda.generarCodigo())
        codigo.append('    push    eax')
        codigo.append(self.derecha.generarCodigo())
        codigo.append('    mov    ebx, eax')
        codigo.append('    pop    eax')
        if self.operador[1] == '+':
            codigo.append('    add    eax, ebx')
        return '\n'.join(codigo)

    def optimizar(self, constantes=None):
        if constantes is None: constantes = {}
        
        if hasattr(self.izquierda, 'optimizar'):
            self.izquierda = self.izquierda.optimizar(constantes)
        if hasattr(self.derecha, 'optimizar'):
            self.derecha = self.derecha.optimizar(constantes)

        if isinstance(self.izquierda, NodoNumero) and isinstance(self.derecha, NodoNumero):
            izq = float(self.izquierda.valor[1]) if '.' in str(self.izquierda.valor[1]) else int(self.izquierda.valor[1])
            der = float(self.derecha.valor[1]) if '.' in str(self.derecha.valor[1]) else int(self.derecha.valor[1])
            
            # Operaciones aritméticas y relacionales
            if self.operador[1] == '+': valor = izq + der
            elif self.operador[1] == '-': valor = izq - der
            elif self.operador[1] == '*': valor = izq * der
            elif self.operador[1] == '/': valor = izq / der
            elif self.operador[1] == '>': valor = 1 if izq > der else 0 # 1 es True, 0 es False
            elif self.operador[1] == '<': valor = 1 if izq < der else 0
            else:
                return self
                
            str_valor = str(int(valor)) if valor == int(valor) else str(valor)
            return NodoNumero(('NUMBER', str_valor))

        # Simplificación algebraica (valores neutros)
        if self.operador[1] == '*' and isinstance(self.derecha, NodoNumero) and float(self.derecha.valor[1]) == 1:
            return self.izquierda
        if self.operador[1] == '*' and isinstance(self.izquierda, NodoNumero) and float(self.izquierda.valor[1]) == 1:
            return self.derecha
        if self.operador[1] == '+' and isinstance(self.derecha, NodoNumero) and float(self.derecha.valor[1]) == 0:
            return self.izquierda
        if self.operador[1] == '+' and isinstance(self.izquierda, NodoNumero) and float(self.izquierda.valor[1]) == 0:
            return self.derecha

        return self
        
                

class NodoRetorno(NodoAST):
    def __init__(self, expresion): self.expresion = expresion

    def traducir(self):
        return f"return {self.expresion.traducir()}"
    
    def generarCodigo(self): return self.expresion.generarCodigo()

class NodoIdentificador(NodoAST):
    def __init__(self, nombre): self.nombre = nombre
    def generarCodigo(self):
        return f'\n    mov eax [{self.nombre[1]}]'

    def optimizar(self, constantes=None):
        if constantes is not None and self.nombre[1] in constantes:
            return NodoNumero(('NUMBER', str(constantes[self.nombre[1]])))
        return self

class NodoNumero(NodoAST):
    def __init__(self, valor): self.valor = valor
        
    def generarCodigo(self):
        return f'\n    mov eax, {self.valor[1]}'

    def optimizar(self, constantes=None):
        return self

class NodoSi(NodoAST):
    def __init__(self, condicion, cuerpo):
        self.condicion = condicion
        self.cuerpo = cuerpo
        
    def generarCodigo(self):
        # Implementación pendiente para ensamblador
        return "; Instruccion IF no implementada en ASM aun"

    def optimizar(self, constantes=None):
        if constantes is None: constantes = {}
        
        if hasattr(self.condicion, 'optimizar'):
            self.condicion = self.condicion.optimizar(constantes)
            
        # Si la condición se resolvió a un número fijo (1 = True, 0 = False)
        if isinstance(self.condicion, NodoNumero):
            es_verdadero = float(self.condicion.valor[1]) != 0
            if es_verdadero:
                # Eliminación de rama: devolvemos directamente el cuerpo optimizado (una lista)
                cuerpo_opt = []
                for instr in self.cuerpo:
                    if hasattr(instr, 'optimizar'):
                        res = instr.optimizar(constantes)
                        if isinstance(res, list): cuerpo_opt.extend(res)
                        else: cuerpo_opt.append(res)
                    else: cuerpo_opt.append(instr)
                return cuerpo_opt
            else:
                # Si fuera falso, devolvemos una lista vacía (se elimina todo)
                return []
        
        # Si no se pudo evaluar, solo optimizamos adentro
        for i in range(len(self.cuerpo)):
            if hasattr(self.cuerpo[i], 'optimizar'):
                self.cuerpo[i] = self.cuerpo[i].optimizar(constantes)
        return self

# ---------------- Parser ----------------
class Parser:
    def __init__(self, tokens): 
        self.tokens = tokens; self.pos = 0
        
    def obtener_token_actual(self): 
        return self.tokens[self.pos] if self.pos < len(self.tokens) else None
    
    def coincidir(self, tipo_esperado):
        token = self.obtener_token_actual()
        if token and token[0] == tipo_esperado: 
            self.pos += 1; return token
        else: 
            raise SyntaxError(f'Error sintactico: se esperaba {tipo_esperado}, pero se encontro: {token}')

    def parsear(self): 
        funciones = []
        while self.obtener_token_actual() is not None:
            funciones.append(self.funcion())
        return NodoPrograma(funciones)

    def funcion(self):
        # 'inicio ... fin'
        self.coincidir('KEYWORD') # Captura 'inicio'
        
        parametros = []
        cuerpo = self.cuerpo()
        
        self.coincidir('KEYWORD') # Captura 'fin'
        
        return NodoFuncion(('KEYWORD', 'void'), ('IDENTIFIER', 'inicio'), parametros, cuerpo)

    def parametros(self):
        lista = []
        token = self.obtener_token_actual()
        if token and token[1] == ')': return lista
        tipo = self.coincidir('KEYWORD')
        nombre = self.coincidir('IDENTIFIER')
        lista.append(NodoParametros(tipo, nombre))
        while self.obtener_token_actual()[1] and self.obtener_token_actual()[1] == ',':
            self.coincidir('DELIMITER')
            tipo = self.coincidir('KEYWORD'); nombre = self.coincidir('IDENTIFIER')
            lista.append(NodoParametros(tipo, nombre))
        return lista

    def cuerpo(self):
        instrucciones = []
        # El cuerpo termina cuando encontramos 'fin', 'finsi' o '}'
        while self.obtener_token_actual() and self.obtener_token_actual()[1] not in ['fin', 'finsi', '}']:
            token = self.obtener_token_actual()
            if token[1] == 'return': 
                instrucciones.append(self.retorno())
            elif token[1] == 'escribir': 
                instrucciones.append(self.sentencia_imprimir_cpp())
            elif token[1] == 'si': 
                instrucciones.append(self.sentencia_si())
            else: 
                instrucciones.append(self.asignacion())
        return instrucciones

    def sentencia_imprimir_cpp(self):
        token = self.obtener_token_actual()
        nueva_linea = (token[1] == 'escribir')
        self.coincidir('KEYWORD') 
        self.coincidir('DELIMITER') # (
        args = self.lista_argumentos()
        self.coincidir('DELIMITER') # )
        
        return NodoPrint(args, nueva_linea)

    def lista_argumentos(self):
        args = []
        if self.obtener_token_actual()[1] != ')':
            sigue = True
            while sigue:
                args.append(self.expresion())
                if self.obtener_token_actual()[1] == ',': self.coincidir('DELIMITER')
                else: sigue = False
        return args

    def asignacion(self):
        # Pseudocódigo: a = 10 
        nombre = self.coincidir('IDENTIFIER')
        self.coincidir('OPERATOR') # '='
        exp = self.expresion()
        
        return NodoAsignacion(('KEYWORD', 'int'), nombre, exp)

    def retorno(self):
        self.coincidir('KEYWORD'); exp = self.expresion(); self.coincidir('DELIMITER')
        return NodoRetorno(exp)

    def expresion(self):
        izq = self.termino()
        while self.obtener_token_actual() and self.obtener_token_actual()[0] == 'OPERATOR':
            op = self.coincidir('OPERATOR'); der = self.termino()
            izq = NodoOperacion(izq, op, der)
        return izq

    def termino(self):
        token = self.obtener_token_actual()
        if token[0] == 'NUMBER': return NodoNumero(self.coincidir('NUMBER'))
        elif token[0] == 'STRING': return NodoString(self.coincidir('STRING'))
        elif token[0] == 'IDENTIFIER': 
            identificador = self.coincidir('IDENTIFIER')
            siguiente = self.obtener_token_actual()
            if siguiente and siguiente[1] == '(':
                self.coincidir('DELIMITER')
                args = self.lista_argumentos()
                self.coincidir('DELIMITER')
                return NodoLlamada(identificador, args)
            return NodoIdentificador(identificador)
        raise SyntaxError(f"Token inesperado: {token}")

    def sentencia_si(self):
        # manejar: si (condicion) entonces cuerpo finsi
        self.coincidir('KEYWORD') # si
        self.coincidir('DELIMITER') # (
        condicion = self.expresion()
        self.coincidir('DELIMITER') # )
        self.coincidir('KEYWORD') # entonces
        
        cuerpo_si = self.cuerpo()
        
        self.coincidir('KEYWORD') # finsi
        return NodoSi(condicion, cuerpo_si)

Overwriting sintactico_ast.py
